# Data Exploration (Simple Version)
A beginner-friendly walkthrough of the raw CEMS data.  
Same findings as `01_exploration.ipynb`, just simpler code.

---

In [23]:
import pandas as pd
import numpy as np

# load all 5 datasets
raw       = pd.read_csv('../datasets/full/raw_cems_data.csv')
sensors   = pd.read_csv('../datasets/full/sensor_master.csv')
maint     = pd.read_csv('../datasets/full/maintenance_logs.csv')
manual    = pd.read_csv('../datasets/full/manual_entries.csv')
thresholds = pd.read_csv('../datasets/full/regulatory_thresholds.csv')

print('Loaded!')
print(f'raw_cems_data  : {raw.shape}')
print(f'sensor_master  : {sensors.shape}')
print(f'maintenance_logs: {maint.shape}')
print(f'manual_entries : {manual.shape}')
print(f'reg_thresholds : {thresholds.shape}')

Loaded!
raw_cems_data  : (27121, 11)
sensor_master  : (10, 9)
maintenance_logs: (30, 5)
manual_entries : (50, 5)
reg_thresholds : (6, 3)


## 1. First Look

In [24]:
# what columns do we have and what are their types?
raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27121 entries, 0 to 27120
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Record_ID        27121 non-null  object 
 1   Plant_ID         27121 non-null  object 
 2   Stack_ID         27121 non-null  object 
 3   Flow_Rate_m3_hr  21682 non-null  float64
 4   TS               27121 non-null  object 
 5   PM2.5            26726 non-null  object 
 6   SO2              26705 non-null  object 
 7   NOx              26701 non-null  object 
 8   Unit             27121 non-null  object 
 9   Status           26999 non-null  object 
 10  Lat_Lon          27121 non-null  object 
dtypes: float64(1), object(10)
memory usage: 2.3+ MB


In [25]:
# first few rows
raw.head(15)

,Record_ID,Plant_ID,Stack_ID,Flow_Rate_m3_hr,TS,PM2.5,SO2,NOx,Unit,Status,Lat_Lon
0,E00001,PL-01,S-01,2033.5,01-01-2025 00:00,170.3,119.3,103.4,ug/m3,down,"13.083,80.2702"
1,E00002,PL-01,S-02,3377.9,01-01-2025 00:00,<2.0,94.2,31.6,µg/m³,MAINT,"13.0839,80.2721"
2,E00003,PL-02,S-01,3798.4,01-01-2025 00:00,123.9,NaN,101.3,mg/Nm3,FAULT,"19.0763,72.8777"
3,E00004,PL-03,S-01,3121.2,01-01-2025 00:00,117.8,147.1,100.8,ug/m3,OK,"12.9719,77.5955"
4,E00005,PL-03,S-02,3700.5,31-12-2024 24:00,37.0,15.0,157.1,mg/Nm3,FAULT,"12.9722,77.5941"
5,E00006,PL-03,S-03,1117.8,01-01-2025 00:00,197.4,48.6,111.6,ug/m3,FAULT,"12.9726,77.5939"
6,E00007,LOC-DEL-01,A-01,0.0,01-01-2025 00:00,111.9,207.0,132.8,ug/m3,DOWN,"28.6138,77.2087"
7,E00008,LOC-DEL-02,A-02,NaN,01-01-2025 00:00,187.8,159.2,147.2,ug/m3,OK,"28.6209,77.2149"
8,E00009,LOC-MUM-01,A-01,0.0,01-01-2025 00:00,196.5,80.0,124.5,ug/m3,DOWN,"19.0757,72.877"
9,E00010,LOC-BLR-01,A-01,NaN,01-01-2025 00:00,128.8,198.3,125.9,ug/m3,OK,"12.9717,77.5954"


## 2. Null Values
Which columns have missing data?

In [26]:
# count of null values per column
print(raw.isnull().sum())
print(f'\nTotal rows: {len(raw)}')

Record_ID             0
Plant_ID              0
Stack_ID              0
Flow_Rate_m3_hr    5439
TS                    0
PM2.5               395
SO2                 416
NOx                 420
Unit                  0
Status              122
Lat_Lon               0
dtype: int64

Total rows: 27121


In [27]:
# also check for empty strings (these show up as non-null but are still missing)
for col in raw.columns:
    empty_count = (raw[col].astype(str).str.strip() == '').sum()
    if empty_count > 0:
        print(f'{col}: {empty_count} empty strings')

## 3. Timestamps
The `TS` column should be in a consistent format. Let's see what we actually have.

In [28]:
# look at some unique timestamp patterns
print('Sample timestamps:')
for ts in raw['TS'].sample(20, random_state=42):
    print(f'  "{ts}"')

Sample timestamps:
  "29-01-2025 06:15"
  "19-01-2025 08:45"
  "11-01-2025 18:30"
  "08-01-2025 08:15"
  "29-01-2025 19:45"
  "01-01-2025 22:45"
  "10-01-2025 05:15"
  "13-01-2025 17:00"
  "26-01-2025 16:00"
  "20-01-2025 01:45"
  "18-01-2025 08:00"
  "05-01-2025 01:00"
  "07-01-2025 00:15 UTC"
  "16-01-2025 17:45"
  "12-01-2025 11:45"
  "08-01-2025 00:15"
  "15-01-2025 01:45"
  "10-01-2025 12:45"
  "04-01-2025 04:30"
  "20-01-2025 20:30"


In [29]:
# count: how many have hour=24?
has_24 = raw['TS'].str.contains('24:', na=False)
print(f'Hour=24 timestamps: {has_24.sum()}')

# count: how many have UTC?
has_utc = raw['TS'].str.contains('UTC', na=False)
print(f'UTC timestamps: {has_utc.sum()}')

# count: how many use slash format (DD/MM/YYYY)?
has_slash = raw['TS'].str.contains('/', na=False)
print(f'Slash format: {has_slash.sum()}')

# count: how many are ISO format (starts with year like 2025-)?
has_iso = raw['TS'].str.match(r'^\d{4}-', na=False)
print(f'ISO format: {has_iso.sum()}')

# the rest are standard DD-MM-YYYY
standard = ~has_24 & ~has_utc & ~has_slash & ~has_iso
print(f'Standard format: {standard.sum()}')
print(f'\nTotal: {len(raw)}')

Hour=24 timestamps: 797
UTC timestamps: 1292
Slash format: 1329
ISO format: 545
Standard format: 23158

Total: 27121


In [30]:
# show examples of each
if has_24.any():
    print('Hour=24 examples:', raw.loc[has_24, 'TS'].head(3).tolist())
if has_utc.any():
    print('UTC examples:', raw.loc[has_utc, 'TS'].head(3).tolist())
if has_slash.any():
    print('Slash examples:', raw.loc[has_slash, 'TS'].head(3).tolist())
if has_iso.any():
    print('ISO examples:', raw.loc[has_iso, 'TS'].head(3).tolist())

Hour=24 examples: ['31-12-2024 24:00', '31-12-2024 24:15', '31-12-2024 24:45']
UTC examples: ['31-12-2024 19:00 UTC', '31-12-2024 19:00 UTC', '31-12-2024 19:15 UTC']
Slash examples: ['01/01/2025 00:30', '01/01/2025 00:45', '01/01/2025 00:45']
ISO examples: ['2025-01-01 01:45:00', '2025-01-01 02:00:00', '2025-01-01 03:00:00']


## 4. Pollutants (PM2.5, SO2, NOx)
These should be numbers, but let's check what's actually in there.

In [31]:
# check the data types — if 'object', there are strings mixed in
for col in ['PM2.5', 'SO2', 'NOx']:
    print(f'{col} → dtype: {raw[col].dtype}')

PM2.5 → dtype: object
SO2 → dtype: object
NOx → dtype: object


In [32]:
# try converting to numbers — anything that fails is a string issue
for col in ['PM2.5', 'SO2', 'NOx']:
    numeric = pd.to_numeric(raw[col], errors='coerce')  # non-numbers become NaN
    
    # how many couldn't be converted?
    failed = numeric.isna() & raw[col].notna() & (raw[col].astype(str).str.strip() != '')
    
    # how many are negative?
    negatives = (numeric < 0).sum()
    
    # how many are suspiciously high (spikes)?
    spikes = (numeric > 5000).sum()
    
    # how many are empty/null?
    empty = (raw[col].astype(str).str.strip() == '').sum() + raw[col].isna().sum()
    
    print(f'\n{col}:')
    print(f'  Non-numeric strings (BDL etc): {failed.sum()}')
    print(f'  Negative values:               {negatives}')
    print(f'  Spikes (>5000):                {spikes}')
    print(f'  Empty/null:                    {empty}')


PM2.5:
  Non-numeric strings (BDL etc): 1391
  Negative values:               1388
  Spikes (>5000):                805
  Empty/null:                    395

SO2:
  Non-numeric strings (BDL etc): 1386
  Negative values:               1380
  Spikes (>5000):                876
  Empty/null:                    416

NOx:
  Non-numeric strings (BDL etc): 1393
  Negative values:               1386
  Spikes (>5000):                835
  Empty/null:                    420


In [33]:
# what do the non-numeric strings look like?
for col in ['PM2.5', 'SO2', 'NOx']:
    numeric = pd.to_numeric(raw[col], errors='coerce')
    non_numeric = raw.loc[numeric.isna() & raw[col].notna() & (raw[col].astype(str).str.strip() != ''), col]
    if len(non_numeric) > 0:
        print(f'{col} non-numeric samples: {non_numeric.head(5).tolist()}')

PM2.5 non-numeric samples: ['<2.0', '< 5.0', '<2.0', '< 5.0', 'BDL']
SO2 non-numeric samples: ['< 4.0', 'BDL', '< 5.0', '<4.0', '<5.0']
NOx non-numeric samples: ['< 5.0', 'BDL', '<2.0', 'BDL', '< 5.0']


## 5. Unit Column

In [34]:
# what unit values are in the data?
print(raw['Unit'].value_counts())

Unit
ug/m3     21753
mg/Nm3     2745
µg/m³      2623
Name: count, dtype: int64


## 6. Status Column

In [35]:
# what status values are in the data?
print(raw['Status'].value_counts())

Status
FAULT          6308
OK             6276
MAINT          6239
error           630
  MAINT         616
offline         606
 maint          601
 fault          595
DOWN            593
Error 404       585
  OK            577
ok              575
maintenance     574
 ok             560
OFFLINE         556
 Ok             554
down            554
Name: count, dtype: int64


## 7. Lat/Lon

In [36]:
# check for obvious issues: semicolons, impossible values
has_semicolon = raw['Lat_Lon'].str.contains(';', na=False)
has_negative999 = raw['Lat_Lon'].str.contains('-999', na=False)

print(f'Semicolon separator: {has_semicolon.sum()}')
print(f'Impossible coords (-999): {has_negative999.sum()}')

if has_semicolon.any():
    print(f'\nSemicolon examples: {raw.loc[has_semicolon, "Lat_Lon"].head(3).tolist()}')
if has_negative999.any():
    print(f'Impossible examples: {raw.loc[has_negative999, "Lat_Lon"].head(3).tolist()}')

Semicolon separator: 178
Impossible coords (-999): 166

Semicolon examples: ['13.083;80.2712', '28.6139;77.209', '13.083;80.2712']
Impossible examples: ['13.0827,-999.0', '28.6139,-999.0', '12.972,-999.0']


## 8. Record ID Gaps

In [37]:
# extract the number from Record_ID (E00001 → 1)
record_nums = raw['Record_ID'].str.replace('E', '').astype(int)

# what's the expected range?
expected = set(range(record_nums.min(), record_nums.max() + 1))
actual = set(record_nums)
missing = expected - actual

print(f'Expected IDs: {len(expected)}')
print(f'Actual IDs:   {len(actual)}')
print(f'Missing IDs:  {len(missing)} (these are gaps that need filling)')

if missing:
    sample = sorted(list(missing))[:10]
    print(f'First 10 missing: {[f"E{x:05d}" for x in sample]}')

Expected IDs: 28800
Actual IDs:   27121
Missing IDs:  1679 (these are gaps that need filling)
First 10 missing: ['E00011', 'E00013', 'E00090', 'E00141', 'E00144', 'E00145', 'E00158', 'E00164', 'E00185', 'E00211']


## 9. Duplicate Check

In [38]:
# any duplicate rows based on Plant_ID + TS?
dupes = raw.duplicated(subset=['Plant_ID', 'TS'], keep=False)
print(f'Duplicate Plant_ID + TS combos: {dupes.sum()}')

if dupes.any():
    print(raw[dupes].head(10))

Duplicate Plant_ID + TS combos: 11043
   Record_ID Plant_ID Stack_ID  Flow_Rate_m3_hr                TS  PM2.5  \
0     E00001    PL-01     S-01           2033.5  01-01-2025 00:00  170.3   
1     E00002    PL-01     S-02           3377.9  01-01-2025 00:00   <2.0   
3     E00004    PL-03     S-01           3121.2  01-01-2025 00:00  117.8   
4     E00005    PL-03     S-02           3700.5  31-12-2024 24:00   37.0   
5     E00006    PL-03     S-03           1117.8  01-01-2025 00:00  197.4   
11    E00014    PL-03     S-01           1537.3  01-01-2025 00:15  103.0   
12    E00015    PL-03     S-02           2640.2  01-01-2025 00:15   70.1   
13    E00016    PL-03     S-03           4669.4  01-01-2025 00:15  172.1   
21    E00024    PL-03     S-01           4745.4  01-01-2025 00:30  202.3   
23    E00026    PL-03     S-03           1075.1  01-01-2025 00:30   79.6   

      SO2     NOx    Unit    Status          Lat_Lon  
0   119.3   103.4   ug/m3      down   13.083,80.2702  
1    94.2    31

## 10. Reference Tables

In [39]:
# sensor_master — the lookup table
print('=== Sensor Master (10 sensors) ===')
print(sensors.to_string(index=False))

=== Sensor Master (10 sensors) ===
  Plant_ID Stack_ID Source_Type Sector  Zero_Drift  Span_Mult  LOD_PM25  LOD_SO2  LOD_NOx
     PL-01     S-01       Stack Cement         0.5       1.02       2.0      4.0      5.0
     PL-01     S-02       Stack Cement         0.2       0.98       2.0      4.0      5.0
     PL-02     S-01       Stack  Steel        -0.5       1.05       2.0      5.0      5.0
     PL-03     S-01       Stack  Power         0.8       1.10       5.0      5.0      5.0
     PL-03     S-02       Stack  Power         0.1       1.00       5.0      5.0      5.0
     PL-03     S-03       Stack  Power         0.0       1.01       5.0      5.0      5.0
LOC-DEL-01     A-01     Ambient   City         0.1       1.00       1.0      2.0      2.0
LOC-DEL-02     A-02     Ambient   City         0.0       0.99       1.0      2.0      2.0
LOC-MUM-01     A-01     Ambient   City        -0.1       1.02       1.0      2.0      2.0
LOC-BLR-01     A-01     Ambient   City         0.2       1.00    

In [40]:
# maintenance_logs — when sensors were being serviced
print('=== Maintenance Logs ===')
print(maint.to_string(index=False))

=== Maintenance Logs ===
  Plant_ID Stack_ID      Maint_Start        Maint_End         Technician
LOC-DEL-02     A-02 21-01-2025 23:00 22-01-2025 00:00       Suresh Patel
     PL-01     S-02 02-01-2025 00:00 02-01-2025 01:00             Team B
     PL-01     S-01 05-01-2025 11:00 05-01-2025 13:00       Suresh Patel
     PL-01     S-02 23-01-2025 13:00 23-01-2025 16:00       Vikram Singh
     PL-03     S-01 24-01-2025 07:00 24-01-2025 09:00             Team B
LOC-MUM-01     A-01 21-01-2025 23:00 22-01-2025 03:00             Team C
LOC-DEL-02     A-02 02-01-2025 19:00 02-01-2025 23:00 Maintenance Crew A
LOC-DEL-02     A-02 06-01-2025 19:00 06-01-2025 20:00       Vikram Singh
LOC-BLR-01     A-01 02-01-2025 09:00 02-01-2025 10:00             Team C
     PL-03     S-01 23-01-2025 13:00 23-01-2025 16:00         Ravi Kumar
     PL-03     S-03 23-01-2025 14:00 23-01-2025 17:00       Priya Sharma
LOC-BLR-01     A-01 22-01-2025 03:00 22-01-2025 06:00             Team C
LOC-MUM-01     A-01 27-01-

In [41]:
# regulatory_thresholds — legal limits
print('=== Legal Limits ===')
print(thresholds.to_string(index=False))

=== Legal Limits ===
Pollutant Source_Type  Legal_Limit_ugm3
    PM2.5     Ambient              60.0
    PM2.5       Stack             150.0
      SO2     Ambient              80.0
      SO2       Stack             200.0
      NOx     Ambient              80.0
      NOx       Stack             400.0


## 11. Manual Entries — QC & PII

In [42]:
# double-entry check: do Entry1 and Entry2 match?
manual['Diff_%'] = abs(manual['Lab_PM25_Entry1'] - manual['Lab_PM25_Entry2']) / manual['Lab_PM25_Entry1'] * 100

print('Lab PM2.5 double-entry comparison:')
print(manual[['Log_ID', 'Lab_PM25_Entry1', 'Lab_PM25_Entry2', 'Diff_%']].to_string(index=False))

qc_fail = manual['Diff_%'] > 1.0
print(f'\nQC Failures (>1% difference): {qc_fail.sum()} out of {len(manual)}')

Lab PM2.5 double-entry comparison:
Log_ID  Lab_PM25_Entry1  Lab_PM25_Entry2     Diff_%
 L0001             98.7            987.0 900.000000
 L0002            156.4            156.7   0.191816
 L0003             49.6             49.2   0.806452
 L0004            123.7            124.2   0.404204
 L0005            132.9            132.8   0.075245
 L0006             69.9            699.0 900.000000
 L0007             86.8             87.1   0.345622
 L0008            149.0            149.3   0.201342
 L0009            122.2            122.0   0.163666
 L0010             44.4             44.5   0.225225
 L0011             17.4            174.0 900.000000
 L0012            149.1            148.9   0.134138
 L0013             90.2             90.1   0.110865
 L0014             49.4             49.0   0.809717
 L0015            112.5            112.4   0.088889
 L0016             74.9             75.2   0.400534
 L0017             66.2             65.8   0.604230
 L0018             19.3      

In [43]:
# PII check: any phone numbers or emails in the notes?
for i, note in enumerate(manual['Inspection_Notes']):
    has_phone = bool(pd.Series([note]).str.contains(r'\b\d{10}\b', regex=True).iloc[0])
    has_email = bool(pd.Series([note]).str.contains(r'\S+@\S+', regex=True).iloc[0])
    
    if has_phone or has_email:
        print(f'PII found in row {i}: "{note[:80]}..."')

total_pii = manual['Inspection_Notes'].str.contains(r'\b\d{10}\b|\S+@\S+', regex=True).sum()
print(f'\nTotal notes with PII: {total_pii}')

PII found in row 4: "Inspection passed. Signed off by Vikram (vikram.s@gov.in, 9090909090)...."
PII found in row 5: "System broken. Call Amit at 9876543210 or amit@email.com...."
PII found in row 9: "System broken. Call Amit at 9876543210 or amit@email.com...."
PII found in row 12: "Sensor drift detected. Contact Priya at 8765432109 for calibration...."
PII found in row 13: "Laser replaced by tech Suresh. Reach him at suresh.p@company.org or 7654321098...."
PII found in row 20: "Laser replaced by tech Suresh. Reach him at suresh.p@company.org or 7654321098...."
PII found in row 22: "Sensor drift detected. Contact Priya at 8765432109 for calibration...."
PII found in row 23: "Recalibrated by vendor. Invoice to anita@vendorcorp.co.in. Mobile: 8080808080...."
PII found in row 25: "Urgent: fan malfunction. Notify ravi.k@gmail.com immediately. Ph: 9988776655...."
PII found in row 33: "System broken. Call Amit at 9876543210 or amit@email.com...."
PII found in row 34: "Inspection passed. Sign

---
## Summary
All the issues we found that need cleaning.

In [44]:
print('ISSUES FOUND:')
print(f'  1. Timestamps with hour=24:        {has_24.sum()}')
print(f'  2. Timestamps with UTC:             {has_utc.sum()}')
print(f'  3. Timestamps with slash format:    {has_slash.sum()}')
print(f'  4. Timestamps in ISO format:        {has_iso.sum()}')
print(f'  5. Non-standard unit values:        {(raw["Unit"] != "ug/m3").sum()}')
print(f'  6. Non-standard status values:      {(~raw["Status"].isin(["OK","FAULT","MAINT"])).sum()}')
print(f'  7. Bad lat/lon coordinates:         {(has_semicolon | has_negative999).sum()}')
print(f'  8. Missing Record_ID gaps:          {len(missing)}')
print(f'  9. Manual entry QC failures:        {qc_fail.sum()}')
print(f' 10. Notes with personal info (PII):  {total_pii}')
print(f'\nNext step: 02_cleaning.ipynb')

ISSUES FOUND:
  1. Timestamps with hour=24:        797
  2. Timestamps with UTC:             1292
  3. Timestamps with slash format:    1329
  4. Timestamps in ISO format:        545
  5. Non-standard unit values:        5368
  6. Non-standard status values:      8298
  7. Bad lat/lon coordinates:         344
  8. Missing Record_ID gaps:          1679
  9. Manual entry QC failures:        10
 10. Notes with personal info (PII):  16

Next step: 02_cleaning.ipynb
